# Sampling SFT — saved workflows

Tickets 01–03 · optional CLI client. Source preparation and plan inspection run on CPU.
Real engine checks run through separately approved RunPod jobs.


## 1. Select the checkout and artifact root

Use the existing artifact tree. For Colab CPU, mount Drive first; set `REPO_DIR` to your checked-out release.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path.cwd().resolve()
if not (REPO_DIR / "pilot_eval").is_dir():
    REPO_DIR = REPO_DIR.parent
if not (REPO_DIR / "pilot_eval").is_dir():
    raise RuntimeError("Set REPO_DIR to the checked-out spar-pilot repository.")

ARTIFACT_ROOT = REPO_DIR / "artifacts"
PLAN_NAME = "sampling-sft-v1"
# In Colab CPU, uncomment before setting ARTIFACT_ROOT:
# from google.colab import drive
# drive.mount("/content/drive")
# ARTIFACT_ROOT = Path("/content/drive/MyDrive/SPAR/spar-pilot/artifacts")


## 2. Select the existing source configurations

Gold may be its original `sft.config.json` or the prepared SFT activation configuration that explicitly points to it.
GRPO uses its existing `activation.prepared.json`. Paths are relative to the artifact root; no directories are guessed from evaluation reports.


In [ ]:
SFT_SOURCE = "plans/pilot2-sft-fp32-l4-batch2-v1/sft.config.json"
GRPO_SOURCE = "plans/pilot4-grpo-fp32-v1-writes-v1/activation.prepared.json"

selection_path = ARTIFACT_ROOT / "plans" / PLAN_NAME / "source-selection.json"
selection = {"sft": SFT_SOURCE, "grpo": GRPO_SOURCE}
selection_path.parent.mkdir(parents=True, exist_ok=True)
if selection_path.exists() and json.loads(selection_path.read_text()) != selection:
    raise RuntimeError("Saved selection differs. Use a new PLAN_NAME to preserve the old plan.")
if not selection_path.exists():
    selection_path.write_text(json.dumps(selection, indent=2) + "\n")

def run_command(operation, *arguments):
    return subprocess.run(
        [sys.executable, "-u", "-m", "pilot_eval", operation,
         "--output-root", str(ARTIFACT_ROOT), *map(str, arguments)],
        cwd=REPO_DIR, check=True,
    )


## 3. Prepare the immutable plan

Saves the selected paths and approved recipe. Existing compatible plans are reused.


In [ ]:
run_command("sampling-sft-prepare", "--selection", selection_path, "--name", PLAN_NAME)
PLAN_PATH = ARTIFACT_ROOT / "plans" / PLAN_NAME / "sampling-sft.plan.json"


## 4. Audit the original sources

Checks the 512 training examples, 150 held-out identities, saved masks/order, original L4 runtime, training completion and all five adapter checkpoints.
Raw historical files are read only. Output is `source-audited` or `blocked`; neither means engine qualification or experiment execution.


In [ ]:
run_command("sampling-sft-audit", "--config", PLAN_PATH)


## 5. Verify and restore

After a reconnect, run cells 1–2, then this cell. The saved plan determines the paths; no generation is restarted.
If source contents or availability changed, preserve this plan and prepare a new named one.


In [ ]:
PLAN_PATH = ARTIFACT_ROOT / "plans" / PLAN_NAME / "sampling-sft.plan.json"
run_command("sampling-sft-status", "--config", PLAN_PATH)
run_command("sampling-sft-verify", "--config", PLAN_PATH)

config = json.loads(PLAN_PATH.read_text())
audit_path = ARTIFACT_ROOT / config["run_path"] / "results" / "source-audit.json"
audit = json.loads(audit_path.read_text())
print("Audit:", audit_path)
print("Sources ready:", audit["source_ready"])
print("Next:", "engine qualification (not implemented yet)" if audit["source_ready"] else "resolve the listed source blockers in a new plan")


## Current boundary

Implemented: source audit, budget/export/stop lifecycle, bounded sampler, T3 score receipts and engine profiles.
Real engine qualification, the exact-resume pilot, full generation and training remain pending.
The primary runbook is `docs/operations/sampling-sft-engine.md`.


## 6. Inspect the shared budget (CPU)

Initializes or restores the one experiment ledger. It reserves no money and starts no job.
The CLI/run-script guide is `docs/operations/sampling-sft-runpod.md`; this notebook is optional.


In [ ]:
run_command("sampling-sft-budget-init", "--config", PLAN_PATH)
run_command("sampling-sft-budget-status")


## 7. Optional offline lifecycle smoke

Disabled by default. Uses a built-in no-op and simulated local copies, with zero cloud cost.
It cannot qualify RunPod, issue paid approval, or stop a real Pod.


In [ ]:
RUN_OFFLINE_LIFECYCLE_SMOKE = False
if RUN_OFFLINE_LIFECYCLE_SMOKE:
    request = {
        "job_id": "lifecycle-demo", "kind": "noop", "stage": "sampling",
        "mode": "noop", "command": [],
        "outputs": [f"runs/sampling-sft/{PLAN_NAME}/lifecycle-demo/result.json"],
    }
    request_path = PLAN_PATH.parent / "demo-request.json"
    request_path.write_text(json.dumps(request, indent=2) + "\n")
    job_dir = ARTIFACT_ROOT / "runs" / "sampling-sft" / PLAN_NAME / "jobs" / "lifecycle-demo"
    job_path = job_dir / "job.plan.json"
    export_dir = ARTIFACT_ROOT / "exports" / PLAN_NAME / "lifecycle-demo"
    run_command("sampling-sft-job-prepare", "--config", PLAN_PATH, "--request", request_path)
    run_command("sampling-sft-job-run", "--job", job_path)
    for role in ("network", "downloaded"):
        run_command("sampling-sft-job-export", "--job", job_path,
                    "--copy-root", export_dir / role, "--role", role,
                    "--receipt", export_dir / f"{role}-receipt.json")
    run_command("sampling-sft-job-finish", "--job", job_path,
                "--network-receipt", export_dir / "network-receipt.json",
                "--download-receipt", export_dir / "downloaded-receipt.json")


## 8. Optional bounded engine plan (CPU)

Supply a reviewed settings file from the engine runbook. This freezes inputs; it starts no model.
Run the resulting plan only through an approved budgeted job, using the CLI run script.


In [ ]:
ENGINE_SETTINGS = ""  # Existing explicit settings JSON; leave empty to skip.
ENGINE_NAME = PLAN_NAME + "-engine-bounded-v1"
if ENGINE_SETTINGS:
    run_command("sampling-sft-instrument-prepare", "--config", PLAN_PATH,
                "--settings", Path(ENGINE_SETTINGS), "--name", ENGINE_NAME)
    ENGINE_PLAN = ARTIFACT_ROOT / "plans" / ENGINE_NAME / "sampling-sft.instrument.json"
    run_command("sampling-sft-instrument-status", "--config", ENGINE_PLAN)
else:
    print("Engine plan skipped. CLI/run-script instructions: docs/operations/sampling-sft-engine.md")
